# Webserv - Guide de Correction

Analyse point par point du sujet de correction vs le code actuel.

Légende : <span style="color: green; font-weight: bold;">FAIT</span> | <span style="color: orange; font-weight: bold;">PARTIEL</span> | <span style="color: red; font-weight: bold;">MANQUANT</span> | <span style="color: dodgerblue; font-weight: bold;">A TESTER</span>

---
## 1. Check the code and ask questions

### Explain the basics of an HTTP server
<span style="color: green; font-weight: bold;">FAIT</span> — Le serveur gère le cycle complet : socket → bind → listen → accept → recv → parse → route → send → close.

### Ask which function they used for I/O Multiplexing
<span style="color: green; font-weight: bold;">FAIT</span> — `poll()` est utilisé (`Server.cpp:579`). Défini dans le Makefile comme `POLL_METHOD = poll` sur Linux.

### Ask to get an explanation of how select (or equivalent) is working
<span style="color: green; font-weight: bold;">FAIT</span> — `poll()` surveille un tableau de `struct pollfd` avec les events POLLIN/POLLOUT. Timeout de 1000ms.

### Ask if they use only one select (or equivalent)
<span style="color: green; font-weight: bold;">FAIT</span> — Un seul appel `poll()` dans la boucle principale `Server::run()` (ligne 579). Il gère à la fois :
- Les sockets serveur (accept)
- Les sockets clients (read/write)
- Les pipes CGI (read)

### The select should check fd for read and write AT THE SAME TIME
<span style="color: green; font-weight: bold;">FAIT</span> — Les sockets clients sont enregistrés avec `pfd.events = POLLIN | POLLOUT` (ligne 147). poll() vérifie les deux en même temps.

### There should be only one read or one write per client per select
<span style="color: green; font-weight: bold;">FAIT</span> — Dans `handleClientEvents()` : un seul `client->readData()` (un seul `recv`) et un seul `client->writeData()` (un seul `send`) par tour de boucle.

### Search for all read/recv/write/send and check error handling
<span style="color: green; font-weight: bold;">FAIT</span> — 
- `recv()` dans `Client::readData()` (Client.cpp:106) : vérifie `< 0` (avec EAGAIN/EWOULDBLOCK) et `== 0` (déconnexion)
- `send()` dans `Client::writeData()` (Client.cpp:163) : vérifie `n > 0`
- `read()` dans `Server::handleCGIEvents()` (Server.cpp:463) : boucle jusqu'à `n == 0`
- Client est supprimé dans `toRemove` quand erreur détectée

### Check if returned value is well checked (checking only -1 or 0 is not good)
<span style="color: orange; font-weight: bold;">PARTIEL</span> —
- `recv()` : vérifie `< 0` ET `== 0` séparément → **OK**
- `send()` : vérifie seulement `n > 0`, ne gère pas `n < 0` (erreur d'envoi) → **A VERIFIER** le retour < 0 n'est pas traité explicitement mais le client sera nettoyé par timeout

### If a check of errno is done after read/recv/write/send → mark to 0
<span style="color: green; font-weight: bold;">FAIT</span> — `errno` est vérifié DANS le if de `recv < 0` : `if (errno == EAGAIN || errno == EWOULDBLOCK)` (Client.cpp:112). C'est correct car on est dans le bloc d'erreur.

### Writing or reading ANY file descriptor without going through the select is FORBIDDEN
<span style="color: orange; font-weight: bold;">PARTIEL</span> —
- Toutes les lectures/écritures clients passent par `poll()` → **OK**
- Les pipes CGI passent par `poll()` (ajoutés dans `buildPollFds()`) → **OK**
- `startCGI()` écrit le body POST dans `pipe_in` SANS passer par poll → **ATTENTION** (mais c'est avant que le pipe soit dans poll, et c'est le pipe_in pas le pipe_out)

### The project must compile without any re-link issue
<span style="color: green; font-weight: bold;">FAIT</span> — `make re` compile proprement avec `-Wall -Wextra -Werror -std=c++98`. Pas de warnings.

---
## 2. Configuration

### HTTP response status codes list — check if any is wrong
<span style="color: green; font-weight: bold;">FAIT</span> — Tous les codes sont standards dans `Dico.hpp` (HttpStatus namespace) :
200, 201, 204, 301, 302, 400, 403, 404, 405, 408, 413, 414, 500, 501, 502, 504

### Setup multiple servers with different port
<span style="color: green; font-weight: bold;">FAIT</span> — `default.conf` a 2 serveurs : port 8080 et 8081. `Server::setup()` crée un socket par serveur.

```bash
curl http://localhost:8080/   # Serveur 1
curl http://localhost:8081/   # Serveur 2
```

### Setup multiple servers with different hostname
<span style="color: orange; font-weight: bold;">PARTIEL</span> — `server_name` est parsé et stocké (`localhost`, `example.com`), MAIS le routing par `Host` header n'est **pas implémenté** dans `Server.cpp`. Le serveur dispatch uniquement par port, pas par hostname.

```bash
# Ce test ne marchera PAS correctement :
curl --resolve example.com:80:127.0.0.1 http://example.com/
```

### Setup default error page
<span style="color: green; font-weight: bold;">FAIT</span> — Les pages d'erreur personnalisées sont configurées (`error_page 404 /errors/404.html`) et servies dans `Server.cpp:359-368` :
```cpp
std::map<int, std::string>::iterator errIt = config->error_pages.find(result.error_code);
if (errIt != config->error_pages.end()) { /* serve custom error page */ }
```

```bash
curl http://localhost:8080/nexiste-pas  # Doit afficher la page 404 custom
```

### Limit the client body
<span style="color: red; font-weight: bold;">MANQUANT</span> — `client_max_body_size` est parsé dans la config (server et location), MAIS **jamais vérifié** lors du parsing de la requête. Le serveur accepte n'importe quelle taille de body.

```bash
# Ce test devrait retourner 413 mais retournera 201 :
curl -X POST -H "Content-Type: plain/text" --data "BODY IS HERE write something shorter or longer than body limit" http://localhost:8080/uploads/test.txt
```

**Où le corriger** : dans `Client::readData()` ou dans `Server::handleClientEvents()` après le parsing, vérifier `req->body.size()` ou `req->content_length` vs `config->max_body_size` (ou `loc->max_body_size`).

### Setup routes in a server to different directories
<span style="color: green; font-weight: bold;">FAIT</span> — La config a des locations avec des `root` différents :
- `/` → `./www`
- `/uploads` → `./www/uploads`
- `/cgi-test` → `./www/cgi-test`

### Setup a default file to search for if you ask for a directory
<span style="color: green; font-weight: bold;">FAIT</span> — `LocationConfig::index` = `"index.html"` par défaut. Dans `routeGET()` : quand c'est un répertoire, cherche `filepath + "/" + loc->index`.

### Setup a list of method accepted for a certain route
<span style="color: green; font-weight: bold;">FAIT</span> — `allowed_methods` parsé et vérifié dans `Router::isMethodAllowed()`. Test :
```bash
curl -X DELETE http://localhost:8080/index.html  # 405 (DELETE non autorisé sur /)
curl -X DELETE http://localhost:8080/uploads/t.txt  # OK (DELETE autorisé sur /uploads)
```

---
## 3. Basic checks

### GET requests → should work
<span style="color: green; font-weight: bold;">FAIT</span> — Fichiers statiques, redirections, autoindex, CGI.
```bash
curl http://localhost:8080/              # 200
curl http://localhost:8080/index.html    # 200
```

### POST requests → should work
<span style="color: green; font-weight: bold;">FAIT</span> — Upload de fichiers + CGI POST.
```bash
curl -X POST http://localhost:8080/uploads/test.txt -d "hello"  # 201
curl -X POST http://localhost:8080/cgi-test/test.py -d "data"   # 200
```

### DELETE requests → should work
<span style="color: green; font-weight: bold;">FAIT</span> — Suppression de fichiers.
```bash
curl -X DELETE http://localhost:8080/uploads/test.txt  # 204
```

### UNKNOWN requests → should not produce any crash
<span style="color: green; font-weight: bold;">FAIT</span> — Le parser rejette les méthodes inconnues (`Request.cpp:87`) : seuls GET, POST, DELETE sont acceptés. Le serveur ferme la connexion sans crash (code 000 côté curl).
```bash
curl -X PUT http://localhost:8080/   # Connexion fermée, pas de crash
curl -X PATCH http://localhost:8080/ # Connexion fermée, pas de crash
```

### For every test the status code must be good
<span style="color: green; font-weight: bold;">FAIT</span> — 25/25 tests passent avec `./tests/run_tests.sh`.

### Upload some file to the server and get it back
<span style="color: green; font-weight: bold;">FAIT</span> —
```bash
curl -X POST http://localhost:8080/uploads/hello.txt -d "Hello World"
curl http://localhost:8080/uploads/hello.txt  # → Hello World
```

---
## 4. Check with a browser

### Use the browser to connect to the server
<span style="color: green; font-weight: bold;">FAIT</span> — `http://localhost:8080/` affiche `index.html`.

### Look at the request header and response header
<span style="color: green; font-weight: bold;">FAIT</span> — Les headers sont visibles dans l'onglet Network du navigateur.
```bash
curl -I http://localhost:8080/  # Voir les headers
```

### It should be compatible to serve a fully static website
<span style="color: green; font-weight: bold;">FAIT</span> — HTML, CSS, JS, images servis avec les bons Content-Type (MimeTypes dans Dico.hpp).

### Try a wrong URL on the server
<span style="color: green; font-weight: bold;">FAIT</span> — `http://localhost:8080/nexiste-pas` → 404 avec page d'erreur custom.

### Try to list a directory
<span style="color: green; font-weight: bold;">FAIT</span> — `http://localhost:8080/uploads/` → autoindex HTML avec tableau.

### Try a redirected URL
<span style="color: green; font-weight: bold;">FAIT</span> — `http://localhost:8080/redirect` → 301 vers Google.

---
## 5. Port issues

### Setup multiple ports and use different websites
<span style="color: green; font-weight: bold;">FAIT</span> — Port 8080 (localhost) et 8081 (example.com) dans `default.conf`. Chaque port a sa propre config.

### Try to setup the same port multiple times → should not work
<span style="color: orange; font-weight: bold;">PARTIEL</span> — Le `ConfigParser` ne vérifie **pas** les ports dupliqués. Mais `bind()` dans `createServerSocket()` échouera si le port est déjà pris, et le serveur affichera une erreur + continuera sans ce port.

**Où le corriger** : ajouter une vérification dans `ConfigParser::validateConfig()` :
```cpp
// Vérifier les ports dupliqués
std::set<int> ports;
for (size_t i = 0; i < _servers.size(); i++) {
    if (!ports.insert(_servers[i].listen_port).second)
        throw std::runtime_error("Duplicate port: " + _servers[i].listen_port);
}
```

### Launch multiple servers with different configs but common ports
<span style="color: orange; font-weight: bold;">PARTIEL</span> — Le serveur gère plusieurs configs sur des ports différents. Si 2 configs ont le même port, le 2e `bind()` échoue mais le serveur ne crash pas (il continue avec les ports qui marchent). Cependant il n'y a pas de routing par `server_name` quand plusieurs configs partagent un port.

---
## 6. Siege & stress test

### Use Siege to run some stress tests
<span style="color: dodgerblue; font-weight: bold;">A TESTER</span> —
```bash
sudo apt install siege
siege -c 50 -t 10s http://localhost:8080/
```

### Availability should be above 99.5%
<span style="color: dodgerblue; font-weight: bold;">A TESTER</span> — Le serveur est non-bloquant (poll + O_NONBLOCK), il devrait tenir. Potentiel problème : pas de limite sur le nombre de clients simultanés (`MAX_CONNECTIONS = 1024` défini mais pas vérifié dans `handleNewConnections()`).

### Check if there is no memory leak
<span style="color: dodgerblue; font-weight: bold;">A TESTER</span> —
```bash
valgrind --leak-check=full --show-leak-kinds=all --track-fds=yes ./webserv config/default.conf
```
Risques potentiels :
- `new Client()` dans `handleNewConnections()` → `delete` dans `toRemove` + destructeur → **OK**
- CGI fork : `waitpid()` appelé dans `finishCGI()`, timeout, et nettoyage client → **OK**
- Pipes CGI : fermés dans `finishCGI()` et dans le nettoyage client → **OK**

### Check if there is no hanging connection
<span style="color: green; font-weight: bold;">FAIT</span> — Timeout client de 60s dans `checkTimeouts()`. Les clients inactifs sont déconnectés.

### You should be able to use siege indefinitely without restarting
<span style="color: dodgerblue; font-weight: bold;">A TESTER</span> — Le serveur boucle sur `poll()` sans allocation croissante (les clients sont nettoyés). Devrait fonctionner mais à valider avec `siege -b`.

---
## 7. Bonus — Cookies and session

<span style="color: red; font-weight: bold;">MANQUANT</span> — Aucun système de cookies/sessions n'est implémenté.

---
## 8. Bonus — CGI

### There's more than one CGI system
<span style="color: green; font-weight: bold;">FAIT</span> — 2 interpréteurs CGI configurés :
- `.py` → `/usr/bin/python3`
- `.sh` → `/usr/bin/bash`

```bash
curl http://localhost:8080/cgi-test/test.py  # Python CGI
curl http://localhost:8080/cgi-test/test.sh  # Bash CGI
```

---
## Résumé

| Section | Status | Détail |
|---------|--------|--------|
| **Check the code** | <span style="color: green; font-weight: bold;">FAIT</span> | poll() unique, read/write via poll, error handling OK |
| **Configuration** | <span style="color: orange; font-weight: bold;">PARTIEL</span> | Manque : `client_max_body_size` non vérifié, virtual hosts par `server_name` non implémenté |
| **Basic checks** | <span style="color: green; font-weight: bold;">FAIT</span> | GET, POST, DELETE, UNKNOWN fonctionnent, upload + retrieval OK |
| **Check with browser** | <span style="color: green; font-weight: bold;">FAIT</span> | Statique, 404, autoindex, redirect OK |
| **Port issues** | <span style="color: orange; font-weight: bold;">PARTIEL</span> | Multi-ports OK, détection port dupliqué manquante, virtual hosts manquant |
| **Siege & stress** | <span style="color: dodgerblue; font-weight: bold;">A TESTER</span> | Architecture non-bloquante en place, timeout OK, à valider avec siege |
| **Bonus: Cookies** | <span style="color: red; font-weight: bold;">MANQUANT</span> | Non implémenté |
| **Bonus: CGI** | <span style="color: green; font-weight: bold;">FAIT</span> | Python + Bash, non-bloquant avec poll() |

---
## Points critiques à corriger 

### 1. `client_max_body_size` (BLOQUANT pour la note)
La correction demande explicitement de tester cette limite. Il faut :
- Après le parsing de la requête, comparer `req->content_length` (ou `req->body.size()`) avec la limite de la config
- Si trop gros → répondre **413 Payload Too Large**
- Fichier à modifier : `Server.cpp` dans `handleClientEvents()`, après `Router::route()` ou dans `Client::readData()`

### 2. Virtual hosts par `server_name` (IMPORTANT pour la note)
Quand 2 serveurs écoutent sur le même port avec des `server_name` différents :
- Lire le header `Host` de la requête
- Trouver le `ServerConfig` dont le `server_name` correspond
- Si aucun ne correspond, utiliser le premier serveur (défaut)
- Fichier à modifier : `Server.cpp` dans `handleClientEvents()`

### 3. Détection port dupliqué (MINEUR)
Ajouter une vérification dans `ConfigParser::validateConfig()` pour refuser les ports en double.

### 4. Gestion `send()` retour < 0 (MINEUR)
Dans `Client::writeData()`, le cas `n < 0` n'est pas traité (seulement `n > 0`).

### 5. Méthodes inconnues → renvoyer 405 au lieu de fermer la connexion (MINEUR)
Actuellement `Request.cpp:87` set `error_code = 405` mais `readData()` retourne -1 → le client est supprimé sans réponse. Il faudrait quand même envoyer la réponse 405 avant de fermer.